### import libraries and Load the cleaned dataset

In [1]:
import pandas as pd

df = pd.read_csv(r"C:\Users\Administrator\Downloads\week7\Intelligent_Complaint_Analysis_for_Financial_Services\data\processed\filtered_complaints.csv")

df.head()


,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,...,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID,has_narrative,narrative_word_count,clean_narrative
0,2025-06-13,Credit card,Store credit card,Getting a credit card,Card opened without my consent or knowledge,A XXXX XXXX card was opened under my name by a...,Company has responded to the consumer and the ...,"CITIBANK, N.A.",TX,78230,...,Consent provided,Web,2025-06-13,Closed with non-monetary relief,Yes,NaN,14069121,True,91,a xxxx xxxx card was opened under my name by a...
1,2025-06-12,Credit card,General-purpose credit card or charge card,"Other features, terms, or problems",Other problem,"Dear CFPB, I have a secured credit card with c...",Company has responded to the consumer and the ...,"CITIBANK, N.A.",NY,11220,...,Consent provided,Web,2025-06-13,Closed with monetary relief,Yes,NaN,14047085,True,156,dear cfpb i have a secured credit card with ci...
2,2025-06-12,Credit card,General-purpose credit card or charge card,Incorrect information on your report,Account information incorrect,I have a Citi rewards cards. The credit balanc...,Company has responded to the consumer and the ...,"CITIBANK, N.A.",IL,60067,...,Consent provided,Web,2025-06-12,Closed with explanation,Yes,NaN,14040217,True,233,i have a citi rewards cards the credit balance...
3,2025-06-09,Credit card,General-purpose credit card or charge card,Problem with a purchase shown on your statement,Credit card company isn't resolving a dispute ...,b'I am writing to dispute the following charge...,Company has responded to the consumer and the ...,"CITIBANK, N.A.",TX,78413,...,Consent provided,Web,2025-06-09,Closed with monetary relief,Yes,NaN,13968411,True,454,b i am writing to dispute the following charge...
4,2025-06-09,Credit card,General-purpose credit card or charge card,Problem when making payments,Problem during payment process,"Although the account had been deemed closed, I...",Company believes it acted appropriately as aut...,Atlanticus Services Corporation,NY,11212,...,Consent provided,Web,2025-06-09,Closed with monetary relief,Yes,NaN,13965746,True,170,although the account had been deemed closed i ...


In [2]:
df["Product"].value_counts()


Product
Credit card    80667
Name: count, dtype: int64

### Create a stratified sample (10,000–15,000 rows)
🎯 Why stratified sampling?

Prevents one product (e.g., Credit Cards) from dominating

Ensures fair representation across all 5 products

Makes your embeddings balanced and reliable

### Define target size

In [3]:
TARGET_SIZE = 12000


### Stratified sampling code

In [4]:
from sklearn.model_selection import train_test_split

sampled_df = (
    df
    .groupby("Product", group_keys=False)
    .apply(lambda x: x.sample(
        n=int(len(x) / len(df) * TARGET_SIZE),
        random_state=42
    ))
)

len(sampled_df)


C:\Users\Administrator\AppData\Local\Temp\ipykernel_16004\2854193613.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(


12000

In [5]:
sampled_df["Product"].value_counts(normalize=True)


Product
Credit card    1.0
Name: proportion, dtype: float64

### Save this result

In [6]:
sampled_df.to_csv(r"C:\Users\Administrator\Downloads\week7\Intelligent_Complaint_Analysis_for_Financial_Services\data\processed/complaints_sampled.csv", index=False)


### Text chunking (VERY IMPORTANT)
❓ Why chunking?

Embedding very long text:

Loses local meaning

Is harder to retrieve

Produces vague results

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter



c:\Users\Administrator\Downloads\week7\Intelligent_Complaint_Analysis_for_Financial_Services\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Define chunking parameters

In [8]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", " ", ""]
)


### Chunk the complaints

In [10]:
documents = []

for _, row in sampled_df.iterrows():
    chunks = text_splitter.split_text(row["clean_narrative"])

    for i, chunk in enumerate(chunks):
        documents.append({
            "text": chunk,
            "complaint_id": row.get("Complaint ID", None),
            "product": row["Product"],
            "chunk_index": i,
            "total_chunks": len(chunks)
        })


### Embedding model

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss
import os

# Load local embedding model 
model = SentenceTransformer(r"C:\Users\Administrator\Downloads\week7\Intelligent_Complaint_Analysis_for_Financial_Services\models\all-MiniLM-L6-v2")

print("Model loaded successfully")


### Generate embeddings for each chunk

In [ ]:
# Extract all text chunks
texts = [doc["text"] for doc in documents]

# Generate embeddings
embeddings = model.encode(texts, show_progress_bar=True, batch_size=64)

print("Embeddings shape:", embeddings.shape)  # Should be (num_chunks, 384)


### Build a FAISS index

In [ ]:
# Dimension of embeddings
embedding_dim = embeddings.shape[1]

# Initialize FAISS index
index = faiss.IndexFlatL2(embedding_dim)  # L2 distance; cosine similarity also possible

# Add embeddings to the index
index.add(np.array(embeddings, dtype=np.float32))



### Save embeddings and metadata

In [ ]:
import pickle

# Save FAISS index
faiss.write_index(index, r"C:\Users\Administrator\Downloads\week7\Intelligent_Complaint_Analysis_for_Financial_Services\vector_store\faiss_index.bin")

# Save metadata separately
metadata_path = r"C:\Users\Administrator\Downloads\week7\Intelligent_Complaint_Analysis_for_Financial_Services\vector_store\metadata.pkl"
with open(metadata_path, "wb") as f:
    pickle.dump(documents, f)
